# Algoritmo ACO — Ant System (TSP)

Implementación del algoritmo de optimización por colonia de hormigas para encontrar una ruta corta que visite todas las ciudades y regrese al inicio.

In [ ]:
import numpy as np
import pandas as pd

# Eric Oswaldo Aguilar Nova
# Algoritmos Metaheuristicos
# 02/11/26
# Algoritmo ACO
# Gabriel Contreras Garcia

print("Eric Oswaldo Aguilar Nova")
print("Algoritmos Metaheuristicos")
print("02/11/26")
print("Algoritmo ACO")
print("Gabriel Contreras Garcia")

# ================= HIPERPARÁMETROS EDITABLES =================
N_ANTS = 10
N_ITERATIONS = 30
ALPHA = 1.0
BETA = 2.0
RHO = 0.20
Q = 75.0
TAU0 = 1.0

N_EJECUCIONES = 10
SEMILLAS = None

# =============================================================

# Diez ciudades fijas para poder comparar ajustes
ciudades = np.array([
    [2, 3], [5, 8], [9, 6], [12, 2], [15, 9],
    [18, 4], [21, 11], [24, 5], [27, 8], [30, 2]
], dtype=float)
nombres = [f"Ciudad {i + 1}" for i in range(len(ciudades))]

In [53]:
def matriz_distancias(coordenadas):
    delta = coordenadas[:, None, :] - coordenadas[None, :, :]
    return np.sqrt(np.sum(delta ** 2, axis=2))


def longitud_ruta(ruta, distancias):
    ciclo = ruta + [ruta[0]]
    return float(sum(distancias[i, j] for i, j in zip(ciclo[:-1], ciclo[1:])))


def ejecutar_aco(coordenadas, n_hormigas, n_iteraciones, alpha, beta,
                 rho, q, tau0, semilla):
    if n_hormigas < 1 or n_iteraciones < 1:
        raise ValueError("N_ANTS y N_ITERATIONS deben ser mayores que cero.")
    if alpha < 0 or beta < 0 or not (0 < rho <= 1) or q <= 0 or tau0 <= 0:
        raise ValueError("ALPHA y BETA >= 0; 0 < RHO <= 1; Q y TAU0 > 0.")

    rng = np.random.default_rng(int(semilla))
    distancias = matriz_distancias(coordenadas)
    n = len(coordenadas)
    if n < 3 or np.any(distancias[np.triu_indices(n, 1)] == 0):
        raise ValueError("Se requieren al menos tres ciudades distintas.")

    heuristica = np.zeros((n, n), dtype=float)
    np.divide(1.0, distancias, out=heuristica, where=distancias > 0)
    feromonas = np.full((n, n), tau0, dtype=float)
    np.fill_diagonal(feromonas, 0.0)

    mejor_ruta = None
    mejor_distancia = float("inf")

    for _ in range(n_iteraciones):
        soluciones = []
        for _ in range(n_hormigas):
            inicio = int(rng.integers(n))
            ruta = [inicio]
            pendientes = set(range(n)) - {inicio}

            while pendientes:
                actual = ruta[-1]
                candidatos = np.array(sorted(pendientes), dtype=int)
                pesos = (feromonas[actual, candidatos] ** alpha
                         * heuristica[actual, candidatos] ** beta)
                total = pesos.sum()
                probabilidades = (
                    pesos / total if np.isfinite(total) and total > 0
                    else np.full(len(candidatos), 1.0 / len(candidatos))
                )
                siguiente = int(rng.choice(candidatos, p=probabilidades))
                ruta.append(siguiente)
                pendientes.remove(siguiente)

            distancia = longitud_ruta(ruta, distancias)
            soluciones.append((ruta, distancia))
            if distancia < mejor_distancia:
                mejor_distancia = distancia
                mejor_ruta = ruta.copy()

        # Evaporación global y depósito de TODAS las hormigas (Ant System)
        feromonas *= (1.0 - rho)
        for ruta, distancia in soluciones:
            aporte = q / distancia
            ciclo = ruta + [ruta[0]]
            for i, j in zip(ciclo[:-1], ciclo[1:]):
                feromonas[i, j] += aporte
                feromonas[j, i] += aporte

    assert sorted(mejor_ruta) == list(range(n))
    return mejor_ruta, mejor_distancia

In [63]:
# Ejecuta esta celda cada vez que quieras una tabla nueva.
if SEMILLAS is None:
    # Nuevas semillas en cada ejecución: variación estocástica real.
    generador_semillas = np.random.default_rng()
    semillas = generador_semillas.choice(
        2**31 - 1, size=N_EJECUCIONES, replace=False
    ).tolist()
else:
    semillas = list(SEMILLAS)
    if len(semillas) != N_EJECUCIONES or len(set(semillas)) != len(semillas):
        raise ValueError("SEMILLAS debe contener N_EJECUCIONES semillas distintas.")

filas = []
for numero, semilla in enumerate(semillas, start=1):
    ruta, distancia = ejecutar_aco(
        coordenadas=ciudades,
        n_hormigas=N_ANTS,
        n_iteraciones=N_ITERATIONS,
        alpha=ALPHA,
        beta=BETA,
        rho=RHO,
        q=Q,
        tau0=TAU0,
        semilla=semilla
    )
    ruta_cerrada = ruta + [ruta[0]]
    filas.append({
        "Ejecución": numero,
        "Semilla": int(semilla),
        "Distancia": round(distancia, 4),
        "Ruta": " → ".join(nombres[i] for i in ruta_cerrada)
    })

resultados = pd.DataFrame(filas)
print("HIPERPARÁMETROS UTILIZADOS")
print(pd.DataFrame([{
    "Hormigas": N_ANTS, "Iteraciones": N_ITERATIONS,
    "Alpha": ALPHA, "Beta": BETA, "Rho": RHO,
    "Q": Q, "Tau0": TAU0
}]).to_string(index=False))
print("\nRESULTADOS POR SEMILLA")
print(resultados.to_string(index=False))
print("\nRESUMEN")
print(f"Mejor distancia: {resultados['Distancia'].min():.4f}")
print(f"Peor distancia:  {resultados['Distancia'].max():.4f}")
print(f"Promedio:        {resultados['Distancia'].mean():.4f}")
print(f"Desviación:      {resultados['Distancia'].std(ddof=0):.4f}")
print("\nNota: las distancias pueden repetirse porque diferentes semillas pueden")
print("converger a la misma ruta; no se fuerza artificialmente ninguna variación.")

HIPERPARÁMETROS UTILIZADOS
 Hormigas  Iteraciones  Alpha  Beta  Rho    Q  Tau0
       10           30    1.0   2.0  0.2 75.0   1.0

RESULTADOS POR SEMILLA
 Ejecución    Semilla  Distancia                                                                                                                     Ruta
         1  181928261    65.9177 Ciudad 10 → Ciudad 9 → Ciudad 7 → Ciudad 5 → Ciudad 3 → Ciudad 2 → Ciudad 1 → Ciudad 4 → Ciudad 6 → Ciudad 8 → Ciudad 10
         2 2144829141    65.9177  Ciudad 8 → Ciudad 10 → Ciudad 9 → Ciudad 7 → Ciudad 5 → Ciudad 3 → Ciudad 2 → Ciudad 1 → Ciudad 4 → Ciudad 6 → Ciudad 8
         3 1120298442    65.9177  Ciudad 5 → Ciudad 7 → Ciudad 9 → Ciudad 10 → Ciudad 8 → Ciudad 6 → Ciudad 4 → Ciudad 1 → Ciudad 2 → Ciudad 3 → Ciudad 5
         4 1667364693    65.9177 Ciudad 10 → Ciudad 9 → Ciudad 7 → Ciudad 5 → Ciudad 3 → Ciudad 2 → Ciudad 1 → Ciudad 4 → Ciudad 6 → Ciudad 8 → Ciudad 10
         5 1347834698    65.9177 Ciudad 10 → Ciudad 9 → Ciudad 7 → Ciudad 5